In [ ]:
pip install torch transformers peft accelerate qwen-vl-utils pillow

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
python -c "import torch; print(f'CUDA 사용 가능: {torch.cuda.is_available()}'); print(f'GPU: {torch.cuda.get_device_name(0)}')"


: 

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel
import torch
from PIL import Image

# 1. 모델 경로 설정 (다운받은 위치로 변경)
adapter_path = "./model/qwen2_vl_finetuned"  # Google Drive나 HF에서 받은 폴더

# 2. 베이스 모델 + LoRA 어댑터 로드
print("🤖 모델 로딩 중...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"  # GPU 자동 할당, CPU만 있으면 "cpu"
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()
model.eval()

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
print("✅ 모델 로딩 완료!")

# 3. 테스트 이미지 경로
test_image = "./img/test.png"  # 테스트할 운송장 이미지 경로

# 4. 추론 함수
def extract_shipping_info(image_path):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": "이 운송장 이미지에서 정보를 추출하여 JSON 형식으로 출력해줘."},
            ],
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)
    
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    
    output = processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    return output


# 5. 실행
result = extract_shipping_info(test_image)
print("\n🎉 추출 결과:")

: 